<a href="https://colab.research.google.com/github/Amanicka2/flyrank_mlengineer/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Amanicka2/flyrank_mlengineer/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In our dataset, each row represents one search query on one webpage for one client on a single day. We use data from the gsc_search_data table and focus strictly on March 2026 for training, saving June 2026 to test the model later. Our goal is to predict if that webpage-query pair will get at least one click over the next 7 days (has_click_next_7d). We leave out same-day conversions (same_day_conversions) on purpose so the model doesn't cheat by looking at future data.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

For our feature bucket, we use past search metrics like 7-day clicks, 7-day impressions, 14-day average position, 7-day click-through rate, and 7-day position movement. For our label bucket, we use a true/false target that shows if a page gets a click in the following week. For our context bucket, we keep identifying information like client ID, page URL, search query, and date. For our excluded bucket, we leave out same-day website conversions because using same-day data introduces data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Query 1: Grain Check (Verify uniqueness of keys)
grain_check = con.sql(f"""
    SELECT COUNT(*) as total_rows, COUNT(DISTINCT (client_hash_id || content_hash_id || report_date)) as unique_keys
    FROM read_parquet('{rel}')
""").df()
print(f"Fact 1 (Grain Check): Total rows = {grain_check['total_rows'][0]:,}, Unique keys = {grain_check['unique_keys'][0]:,}")

# Query 2: Row Count & Date Span
span_check = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('{rel}')
""").df()
print(f"Fact 2 (Row Count & Span): Total Rows = {span_check['total_rows'][0]:,}, Min Date = {span_check['min_date'][0]}, Max Date = {span_check['max_date'][0]}")

# Query 3: Availability Filter
avail_check = con.sql(f"""
    SELECT COUNT(*) as valid_rows
    FROM read_parquet('{rel}')
    WHERE gsc_data_available IS TRUE
""").df()
print(f"Fact 3 (Availability): Rows passing availability check = {avail_check['valid_rows'][0]:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 1 (Grain Check): Total rows = 9,841,378, Unique keys = 9,841,378
Fact 2 (Row Count & Span): Total Rows = 9,841,378, Min Date = 2026-03-01 00:00:00, Max Date = 2026-03-31 00:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 3 (Availability): Rows passing availability check = 3,611,061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset can't show us anything outside of Google Search Console or GA4. Some clients have tons of history while newer ones barely have any, older records are missing GA4 web traffic entirely, and because the data uses 7-day rolling windows, consecutive days end up repeating the exact same trend stats over and over.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query data limits: unbalanced history, missing GA4 early rows, and window overlap
limits_check = con.sql(f"""
    SELECT
        -- 1. Unbalanced history (Min vs Max rows per client)
        MIN(client_rows) AS min_rows_per_client,
        MAX(client_rows) AS max_rows_per_client,

        -- 2. GSC-only early rows (Count rows missing GA4 data)
        COUNT(*) FILTER (WHERE client_has_gsc IS TRUE AND client_has_ga4 IS FALSE) AS gsc_only_rows,

        -- 3. Window overlaps (Total dates in current slice)
        COUNT(DISTINCT report_date) AS total_overlapping_dates
    FROM (
        SELECT *, COUNT(*) OVER(PARTITION BY client_hash_id) AS client_rows
        FROM read_parquet('{rel}')
    )
""").df()

print(limits_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   min_rows_per_client  max_rows_per_client  gsc_only_rows  \
0                  341               988497        3018741   

   total_overlapping_dates  
0                       31  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.